# Encontro 5 - Limpeza e preparação dos dados

**Aluno - prática guiada**

> **Ideia central:** limpar dados é tomar decisões rastreáveis. O código deve preservar a base bruta, justificar cada alteração e provar que o resultado ficou melhor para o objetivo da análise.

Usaremos a mesma base **Automobile (Imports-85)** do encontro anterior. Cada linha representa um automóvel; `price` será tratada como possível variável-alvo.

**Ao final, você deverá conseguir:**

1. tratar valores ausentes de acordo com contexto;
2. localizar duplicidades e valores inválidos;
3. corrigir tipos, nomes e unidades;
4. normalizar variáveis numéricas;
5. criar faixas com *binning*;
6. codificar categorias com *one-hot encoding*;
7. comparar o diagnóstico antes e depois;
8. registrar decisões e possíveis impactos.


## Como usar

Execute as células na ordem com **Shift + Enter**. O notebook procura `dados/automoveis.csv` e, se necessário, usa a URL pública da UCI. A base original ficará em `df_original`; trabalharemos em cópias.

As caixas **Pare e decida** pedem uma justificativa antes do código. Em preparação de dados, a justificativa é parte da resposta.


## 1. Preparação do ambiente e carregamento


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

COLUNAS = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight",
    "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
    "city-mpg", "highway-mpg", "price"
]

candidatos = [Path("dados/automoveis.csv"), Path("automoveis.csv")]
caminho = next((p for p in candidatos if p.exists()), None)

if caminho is not None:
    df_original = pd.read_csv(caminho, na_values="?")
    origem = str(caminho)
else:
    origem = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/autos/imports-85.data"
    )
    df_original = pd.read_csv(origem, names=COLUNAS, na_values="?")

assert df_original.shape == (205, 26)
print("Origem:", origem)
print("Dimensões:", df_original.shape)


## 2. O diagnóstico orienta as decisões


In [ ]:
def diagnostico(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "ausentes": df.isna().sum(),
        "ausentes_%": df.isna().mean().mul(100).round(2),
        "distintos": df.nunique(dropna=False),
    }).sort_values(["ausentes", "distintos"], ascending=[False, True])

diagnostico_antes = diagnostico(df_original)
diagnostico_antes.head(12)


In [ ]:
print("Linhas duplicadas:", df_original.duplicated().sum())
print()
print("Ausências por variável:")
display(df_original.isna().sum()[lambda s: s > 0].sort_values(ascending=False))


<div style="border-left: 4px solid #3D8DFF; padding: 8px 12px; background: #F6F9FF">
<strong>Pare e decida:</strong> por que `price` ausente pode exigir uma estratégia diferente de `normalized-losses` ausente?
</div>

<details><summary>Sugestão de raciocínio</summary>

Se `price` é o alvo da análise, uma linha sem preço não contribui para uma análise supervisionada de preço. Já `normalized-losses` é um atributo: removê-lo ou remover todas as linhas em que falta pode desperdiçar muita informação. A decisão depende do objetivo.

</details>


## 3. Valores ausentes: decidir antes de preencher


In [ ]:
df_trabalho = df_original.copy()

resumo_ausencias = pd.DataFrame({
    "quantidade": df_trabalho.isna().sum(),
    "percentual": df_trabalho.isna().mean().mul(100).round(2),
})
resumo_ausencias[resumo_ausencias["quantidade"] > 0]


### 3.1 Remover observações sem o alvo

Há apenas quatro linhas sem `price` (1,95%). Como o exercício prepara dados para analisar preço, removeremos essas observações. Isso não é uma regra universal: é uma decisão ligada ao objetivo.


In [ ]:
linhas_antes = len(df_trabalho)
df_trabalho = df_trabalho.dropna(subset=["price"]).copy()

print("Linhas removidas:", linhas_antes - len(df_trabalho))
print("Linhas restantes:", len(df_trabalho))


### 3.2 Imputar atributos numéricos

Usaremos a **mediana** como regra didática para os atributos numéricos ausentes. Ela é menos sensível a valores extremos que a média. Em um projeto real, compararíamos alternativas e validaríamos o impacto.


In [ ]:
numericas_para_imputar = [
    "normalized-losses", "bore", "stroke", "horsepower", "peak-rpm"
]

medianas_usadas = {}
for coluna in numericas_para_imputar:
    mediana = df_trabalho[coluna].median()
    medianas_usadas[coluna] = mediana
    df_trabalho[coluna] = df_trabalho[coluna].fillna(mediana)

pd.Series(medianas_usadas, name="mediana_usada")


### 3.3 Imputar uma categoria


In [ ]:
moda_portas = df_trabalho["num-of-doors"].mode()[0]
df_trabalho["num-of-doors"] = df_trabalho["num-of-doors"].fillna(moda_portas)

print("Moda usada em num-of-doors:", moda_portas)
print("Ausências restantes:", int(df_trabalho.isna().sum().sum()))


<div style="border-left: 4px solid #3D8DFF; padding: 8px 12px; background: #F6F9FF">
<strong>Cuidado:</strong> média, mediana e moda não recuperam o valor verdadeiro. Toda imputação introduz uma hipótese e pode reduzir variabilidade ou distorcer relações.
</div>


## 4. Duplicidades e valores inválidos


In [ ]:
duplicadas = df_trabalho.duplicated().sum()
print("Duplicadas encontradas:", duplicadas)

df_trabalho = df_trabalho.drop_duplicates().copy()


Encontrar zero duplicidades também é um resultado. Já os valores inválidos podem não ser `NaN`; precisam de regras de domínio.


In [ ]:
exemplo_invalidos = pd.DataFrame({
    "idade": [24, -4, 31],
    "nota": [8.5, 15.0, 7.0],
    "frequencia_pct": [92, 132, 80],
})

regras_invalidas = (
    (exemplo_invalidos["idade"] < 0)
    | (~exemplo_invalidos["nota"].between(0, 10))
    | (~exemplo_invalidos["frequencia_pct"].between(0, 100))
)

exemplo_invalidos.assign(invalido=regras_invalidas)


In [ ]:
assert (df_trabalho["price"] > 0).all(), "Há preço não positivo."
assert (df_trabalho["city-mpg"] > 0).all(), "Há consumo inválido."
assert df_trabalho["num-of-doors"].isin(["two", "four"]).all()

print("Regras de domínio verificadas.")


## 5. Tipos, nomes e unidades


In [ ]:
categoricas = [
    "make", "fuel-type", "aspiration", "num-of-doors", "body-style",
    "drive-wheels", "engine-location", "engine-type",
    "num-of-cylinders", "fuel-system"
]

for coluna in categoricas:
    df_trabalho[coluna] = df_trabalho[coluna].astype("category")

df_trabalho[categoricas].dtypes


O consumo urbano está em milhas por galão (`city-mpg`). Criaremos uma nova coluna em litros por 100 km, mantendo a original para rastreabilidade.


In [ ]:
df_trabalho["city-L/100km"] = 235 / df_trabalho["city-mpg"]

df_trabalho[["city-mpg", "city-L/100km"]].head()


## 6. Normalização: criar escalas comparáveis


In [ ]:
minimo = df_trabalho["length"].min()
maximo = df_trabalho["length"].max()

df_trabalho["length_minmax"] = (
    (df_trabalho["length"] - minimo) / (maximo - minimo)
)

media = df_trabalho["length"].mean()
desvio = df_trabalho["length"].std()
df_trabalho["length_zscore"] = (df_trabalho["length"] - media) / desvio

df_trabalho[["length", "length_minmax", "length_zscore"]].describe().T


<div style="border-left: 4px solid #3D8DFF; padding: 8px 12px; background: #F6F9FF">
<strong>Observe:</strong> normalização não corrige erro e não remove outlier. Ela transforma a escala. O valor original foi preservado para interpretação.
</div>


## 7. Binning: números viram faixas


In [ ]:
limites = [0, 10_000, 20_000, float("inf")]
rotulos = ["baixo", "medio", "alto"]

df_trabalho["faixa_preco"] = pd.cut(
    df_trabalho["price"],
    bins=limites,
    labels=rotulos,
    include_lowest=True,
)

df_trabalho["faixa_preco"].value_counts().sort_index()


<div style="border-left: 4px solid #3D8DFF; padding: 8px 12px; background: #F6F9FF">
<strong>Trade-off:</strong> as faixas facilitam comunicação e segmentação, mas apagam diferenças dentro de cada intervalo. Os limites precisam de justificativa.
</div>


## 8. Categorias em números: one-hot encoding


In [ ]:
df_modelo = pd.get_dummies(
    df_trabalho,
    columns=["fuel-type"],
    prefix="fuel",
    dtype=int,
)

df_modelo[["fuel_gas", "fuel_diesel"]].head()


`gas = 1` e `diesel = 0` representam presença/ausência. Não atribuímos uma ordem artificial. Uma categoria nominal como cor não deve virar `vermelho=1`, `azul=2`, `verde=3` sem um motivo válido.


## 9. Diagnóstico final e registro de decisões


In [ ]:
diagnostico_depois = diagnostico(df_modelo)

comparacao = pd.DataFrame({
    "antes": {
        "linhas": df_original.shape[0],
        "colunas": df_original.shape[1],
        "ausencias": int(df_original.isna().sum().sum()),
        "duplicadas": int(df_original.duplicated().sum()),
    },
    "depois": {
        "linhas": df_modelo.shape[0],
        "colunas": df_modelo.shape[1],
        "ausencias": int(df_modelo.isna().sum().sum()),
        "duplicadas": int(df_modelo.duplicated().sum()),
    },
})

comparacao


In [ ]:
assert df_modelo.shape[0] == 201
assert df_modelo.isna().sum().sum() == 0
assert df_modelo.duplicated().sum() == 0
assert df_modelo["length_minmax"].between(0, 1).all()
assert set(df_modelo["faixa_preco"].dropna().astype(str)) <= {"baixo", "medio", "alto"}

print("Validação final concluída.")
df_modelo.info()


In [ ]:
registro_decisoes = pd.DataFrame([
    ["price", "Remover 4 linhas ausentes", "É o alvo deste exercício", "Reduz a amostra para 201 linhas"],
    ["normalized-losses", "Imputar mediana", "Evitar perder 41 registros", "Pode reduzir a variabilidade"],
    ["bore, stroke, horsepower, peak-rpm", "Imputar mediana", "Poucas ausências numéricas", "Valores imputados não são observados"],
    ["num-of-doors", "Imputar moda", "Duas ausências categóricas", "Reforça a categoria majoritária"],
    ["duplicidades", "Verificar e remover", "Evitar contagem repetida", "Nenhuma duplicata encontrada"],
    ["city-mpg", "Criar city-L/100km", "Facilitar interpretação local", "Coluna original preservada"],
    ["length", "Criar min-max e z-score", "Comparar escalas", "Transformações dependem da amostra"],
    ["price", "Criar faixa_preco", "Segmentação didática", "Perda de precisão dentro da faixa"],
    ["fuel-type", "One-hot encoding", "Representar categoria nominal", "Cria duas colunas binárias"],
], columns=["variavel", "acao", "justificativa", "impacto_possivel"])

registro_decisoes


## 10. Laboratório - sua vez


Crie `df_entrega` a partir de `df_original` e cumpra a missão abaixo. Você pode usar decisões diferentes das demonstradas, desde que sejam tecnicamente válidas e justificadas.

1. mostrar `info()` e ausências antes;
2. identificar duplicidades e valores inválidos;
3. tratar todas as ausências e justificar cada estratégia;
4. corrigir ao menos um tipo ou formato;
5. normalizar uma variável numérica sem apagar a original;
6. criar uma faixa com `pd.cut()`;
7. codificar uma variável categórica com `pd.get_dummies()`;
8. mostrar `info()` e indicadores de qualidade depois;
9. escrever uma célula Markdown chamada **Decisões de preparação**;
10. exportar a base e o registro de decisões.

**Critério principal:** uma pessoa deve conseguir entender o que mudou, por que mudou e que impacto isso pode causar.


In [ ]:
# Comece aqui. Preserve df_original e trabalhe em uma cópia.
df_entrega = df_original.copy()

# 1. Diagnóstico inicial

# 2. Duplicidades e regras de validade

# 3. Valores ausentes

# 4. Tipos, formatos ou unidades

# 5. Normalização

# 6. Binning

# 7. Encoding

# 8. Diagnóstico final


### Decisões de preparação

Substitua este texto por suas decisões. Para cada alteração, registre:

- **o que** mudou;
- **por que** a estratégia foi escolhida;
- **quantos registros** foram afetados;
- **qual impacto** a decisão pode produzir.


In [ ]:
# 9. Exporte sua base e seu registro de decisões.
# df_entrega.to_csv("automoveis_preparados_entrega.csv", index=False)


## Checklist da entrega

- [ ] trabalhei em uma cópia da base original;
- [ ] apresentei diagnóstico antes e depois;
- [ ] tratei ausências com justificativa;
- [ ] verifiquei duplicidades e regras de domínio;
- [ ] corrigi ao menos um tipo, formato ou unidade;
- [ ] normalizei uma variável preservando a original;
- [ ] criei uma faixa com `pd.cut()`;
- [ ] codifiquei uma categoria nominal;
- [ ] registrei impactos possíveis;
- [ ] exportei a base preparada.


## Fechamento

> **Dados limpos não são dados verdadeiros por definição.** São dados transformados por decisões que precisam ser coerentes, testáveis e documentadas.

Na próxima etapa, a base preparada poderá ser explorada e transformada com objetivos analíticos mais específicos.


## Fontes e licença

- Schlimmer, J. (1985). **Automobile** [Dataset]. UCI Machine Learning Repository. DOI: [10.24432/C5B01C](https://doi.org/10.24432/C5B01C). Licença CC BY 4.0.
- IBM Data Science Professional Certificate, curso **Data Analysis with Python**, Module 2 - Data Wrangling: [Coursera](https://www.coursera.org/learn/data-analysis-with-python).

O material adapta conceitos de preparação de dados ao contexto didático da disciplina e acrescenta verificações de duplicidade e validade de domínio.
